# Load Data And Explore Data

Notebook to get an initial feel for the data and it's schema and distributions. Not built on in any of the other files, just used to get some information on what we'll be dealing with.

In [1]:
%pip install datasets huggingface_hub polars pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import polars as pl
from datasets import load_dataset
from itertools import islice
import json

/Users/mhedlund/CIS2450/cis2450-final-project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Conduct initial data profiling for samples from all datasets

In [ ]:
def from_iter_dataset(ds, n: int) -> list[dict]:
    return list(islice(iter(ds), n))


def sample_n_to_polars(ds, n: int) -> pl.DataFrame:
    rows = from_iter_dataset(ds, n)
    return pl.DataFrame(rows)


def initial_profile_data(df: pl.DataFrame, text_col: str | None = None, n_examples: int = 3) -> None:
    print(f"rows: {df.height:,}")
    print(f"cols: {df.width}")

    print("\n" \
    "Schema:")
    for k, v in df.schema.items():
        print(f"  {k}: {v}")

    print("\n" \
    "Null data counts")
    print(df.null_count())

    if text_col and text_col in df.columns:
        text_stats = (
            df.with_columns(
                pl.col(text_col).cast(pl.Utf8).str.len_chars().alias("text_len")
            )
            .select([
                pl.col("text_len").mean().alias("mean_len"),
                pl.col("text_len").median().alias("median_len"),
                pl.col("text_len").min().alias("min_len"),
                pl.col("text_len").max().alias("max_len"),
            ])
        )
        print("\nText stats:")
        print(text_stats)

        print(f"\nExamples from '{text_col}':")
        examples = (
            df.select(text_col)
            .filter(pl.col(text_col).is_not_null())
            .head(n_examples)
        )
        print(examples)

In [ ]:
twitter_stream = load_dataset(
    "enryu43/twitter100m_tweets",
    split="train",
    streaming=True
)
twitter_sample = sample_n_to_polars(twitter_stream, 5000)
twitter_sample.head()

twitter_sample.select([
    pl.col("user").n_unique().alias("unique_users"),
    pl.col("date").min().alias("min_date"),
    pl.col("date").max().alias("max_date"),
    pl.col("likes").mean().alias("mean_likes"),
    pl.col("retweets").mean().alias("mean_retweets"),
])
initial_profile_data(twitter_sample, text_col="tweet")

rows: 5,000
cols: 8

Schema:
  user: String
  id: Int64
  tweet: String
  replies: Int64
  retweets: Int64
  likes: Int64
  quotes: Int64
  date: String

Null data counts
shape: (1, 8)
┌──────┬─────┬───────┬─────────┬──────────┬───────┬────────┬──────┐
│ user ┆ id  ┆ tweet ┆ replies ┆ retweets ┆ likes ┆ quotes ┆ date │
│ ---  ┆ --- ┆ ---   ┆ ---     ┆ ---      ┆ ---   ┆ ---    ┆ ---  │
│ u32  ┆ u32 ┆ u32   ┆ u32     ┆ u32      ┆ u32   ┆ u32    ┆ u32  │
╞══════╪═════╪═══════╪═════════╪══════════╪═══════╪════════╪══════╡
│ 0    ┆ 0   ┆ 0     ┆ 0       ┆ 0        ┆ 0     ┆ 0      ┆ 0    │
└──────┴─────┴───────┴─────────┴──────────┴───────┴────────┴──────┘

Text stats:
shape: (1, 4)
┌──────────┬────────────┬─────────┬─────────┐
│ mean_len ┆ median_len ┆ min_len ┆ max_len │
│ ---      ┆ ---        ┆ ---     ┆ ---     │
│ f64      ┆ f64        ┆ u32     ┆ u32     │
╞══════════╪════════════╪═════════╪═════════╡
│ 134.5152 ┆ 130.0      ┆ 1       ┆ 425     │
└──────────┴────────────┴─────────┴─

We see the general schema for the twitter data we are collecting, including information we want to keep like the post itself in the "tweet" field, but also domain specific information like likes, quotes, and retweets which are twitter specific so we won't keep. We can also see that the data has already had null values removed.

In [ ]:
reddit_stream = load_dataset(
"anhchanghoangsg/reddit_pushshift_dataset_cleaned",
split="train",
streaming=True
)

reddit_sample = sample_n_to_polars(reddit_stream, 5000)
reddit_sample.head()
initial_profile_data(reddit_sample, text_col="body")

rows: 5,000
cols: 10

Schema:
  author: String
  body: String
  created_utc: String
  controversiality: Int64
  score: Int64
  name: String
  parent_id: String
  subreddit: String
  subreddit_id: String
  link_id: String

Null data counts
shape: (1, 10)
┌────────┬──────┬─────────────┬───────────────┬───┬───────────┬───────────┬──────────────┬─────────┐
│ author ┆ body ┆ created_utc ┆ controversial ┆ … ┆ parent_id ┆ subreddit ┆ subreddit_id ┆ link_id │
│ ---    ┆ ---  ┆ ---         ┆ ity           ┆   ┆ ---       ┆ ---       ┆ ---          ┆ ---     │
│ u32    ┆ u32  ┆ u32         ┆ ---           ┆   ┆ u32       ┆ u32       ┆ u32          ┆ u32     │
│        ┆      ┆             ┆ u32           ┆   ┆           ┆           ┆              ┆         │
╞════════╪══════╪═════════════╪═══════════════╪═══╪═══════════╪═══════════╪══════════════╪═════════╡
│ 0      ┆ 0    ┆ 0           ┆ 0             ┆ … ┆ 0         ┆ 0         ┆ 0            ┆ 0       │
└────────┴──────┴─────────────┴────────

Similarly the reddit dataset has both data we'll want like the "body" of the post itself, as well as domain specific data we'll remove, and has also had null values removed.

In [ ]:
hn_stream = load_dataset(
    "open-index/hacker-news",
    split="train",
    streaming=True
)

hn_sample = sample_n_to_polars(hn_stream, 5000)
hn_sample.head()
initial_profile_data(hn_sample)
hn_sample.columns
hn_sample.select([
    pl.col("id").n_unique().alias("unique_ids"),
    pl.col("time").min().alias("min_time"),
    pl.col("time").max().alias("max_time"),
])
hn_sample.group_by("type").len().sort("len", descending=True)

rows: 5,000
cols: 16

Schema:
  id: Int64
  deleted: Int64
  type: Int64
  by: String
  time: Datetime(time_unit='us', time_zone='UTC')
  text: String
  dead: Int64
  parent: Int64
  poll: Int64
  kids: List(Int64)
  url: String
  score: Int64
  title: String
  parts: List(Null)
  descendants: Int64
  words: List(String)

Null data counts
shape: (1, 16)
┌─────┬─────────┬──────┬─────┬───┬───────┬───────┬─────────────┬───────┐
│ id  ┆ deleted ┆ type ┆ by  ┆ … ┆ title ┆ parts ┆ descendants ┆ words │
│ --- ┆ ---     ┆ ---  ┆ --- ┆   ┆ ---   ┆ ---   ┆ ---         ┆ ---   │
│ u32 ┆ u32     ┆ u32  ┆ u32 ┆   ┆ u32   ┆ u32   ┆ u32         ┆ u32   │
╞═════╪═════════╪══════╪═════╪═══╪═══════╪═══════╪═════════════╪═══════╡
│ 0   ┆ 0       ┆ 0    ┆ 0   ┆ … ┆ 0     ┆ 0     ┆ 0           ┆ 0     │
└─────┴─────────┴──────┴─────┴───┴───────┴───────┴─────────────┴───────┘


type,len
i64,u32
2,3329
1,1671


In [7]:
hn_sample.columns
hn_sample.head(3)
hn_sample.group_by("type").len().sort("len", descending=True)

for col in hn_sample.columns:
    print(col)

id
deleted
type
by
time
text
dead
parent
poll
kids
url
score
title
parts
descendants
words


In [8]:
hn_sample.group_by("type").len().sort("len", descending=True)

hn_sample.select([
    pl.col("type"),
    pl.col("title").is_null().sum().alias("title_nulls"),
    pl.col("text").is_null().sum().alias("text_nulls"),
    pl.col("url").is_null().sum().alias("url_nulls"),
])

hn_sample.group_by("type").agg([
    pl.len().alias("n_rows"),
    pl.col("title").is_not_null().sum().alias("nonnull_title"),
    pl.col("text").is_not_null().sum().alias("nonnull_text"),
    pl.col("url").is_not_null().sum().alias("nonnull_url"),
]).sort("n_rows", descending=True)

for t in hn_sample["type"].unique().to_list():
    print(f"\n===== TYPE: {t} =====")
    print(
        hn_sample
        .filter(pl.col("type") == t)
        .select(["id", "type", "title", "text", "url", "by", "time", "score"])
        .head(3)
    )

hn_sample.with_columns(
    pl.col("text").cast(pl.Utf8).str.len_chars().alias("text_len"),
    pl.col("title").cast(pl.Utf8).str.len_chars().alias("title_len"),
).group_by("type").agg([
    pl.len().alias("n_rows"),
    pl.col("text_len").mean().alias("mean_text_len"),
    pl.col("title_len").mean().alias("mean_title_len"),
]).sort("n_rows", descending=True)




===== TYPE: 1 =====
shape: (3, 8)
┌─────┬──────┬────────────────────┬──────┬───────────────────┬─────────┬───────────────────┬───────┐
│ id  ┆ type ┆ title              ┆ text ┆ url               ┆ by      ┆ time              ┆ score │
│ --- ┆ ---  ┆ ---                ┆ ---  ┆ ---               ┆ ---     ┆ ---               ┆ ---   │
│ i64 ┆ i64  ┆ str                ┆ str  ┆ str               ┆ str     ┆ datetime[μs, UTC] ┆ i64   │
╞═════╪══════╪════════════════════╪══════╪═══════════════════╪═════════╪═══════════════════╪═══════╡
│ 1   ┆ 1    ┆ Y Combinator       ┆      ┆ http://ycombinato ┆ pg      ┆ 2006-10-09        ┆ 57    │
│     ┆      ┆                    ┆      ┆ r.com             ┆         ┆ 18:21:51 UTC      ┆       │
│ 2   ┆ 1    ┆ A Student's Guide  ┆      ┆ http://www.paulgr ┆ phyllis ┆ 2006-10-09        ┆ 16    │
│     ┆      ┆ to Startups        ┆      ┆ aham.com/mit.…    ┆         ┆ 18:30:28 UTC      ┆       │
│ 3   ┆ 1    ┆ Woz Interview: the ┆      ┆ http://www.fo

type,n_rows,mean_text_len,mean_title_len
i64,u32,f64,f64
2,3329,308.827576,0.0
1,1671,0.0,46.59246


In [9]:
hn_comments = (
    hn_sample
    .filter(pl.col("type") == 2)
)

hn_comments.head()

hn_std = (
    hn_comments
    .select([
        pl.lit("hackernews").alias("platform"),
        pl.col("id").cast(pl.Utf8).alias("post_id"),
        pl.col("by").cast(pl.Utf8).alias("author"),
        pl.col("text").cast(pl.Utf8).alias("text"),
        pl.col("time").alias("created_at"),
    ])
)

hn_std.head()

platform,post_id,author,text,created_at
str,str,str,str,"datetime[μs, UTC]"
"""hackernews""","""15""","""sama""","""&#34;the rising star of ventur…",2006-10-09 19:51:01 UTC
"""hackernews""","""17""","""pg""","""Is there anywhere to eat on Sa…",2006-10-09 19:52:45 UTC
"""hackernews""","""22""","""pg""","""It's kind of funny that Sevin …",2006-10-10 02:18:22 UTC
"""hackernews""","""23""","""starklysnarky""","""This is interesting, but the l…",2006-10-10 02:30:53 UTC
"""hackernews""","""30""","""spez""","""Stay tuned...""",2006-10-10 15:34:59 UTC


Lastly the hackernews data is a bit unique in that it has a numeric type to designate what kind of post it is. From the sample we only see 1 and 2, but from reading the specifications there can be other numeric types to represent things like comments, polls, briefs, and more. For our data we are interested only in type 1 data which refers to traditional social media posts.